In [ ]:
%load_ext tensorboard

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATASET_FILE = 'jetbot_dataset_2025-12-13_05-59-20.zip'  # <-- change if needed
DATASET_DIR = 'dataset'
DATASET_ZIP = os.path.join(DATASET_DIR, DATASET_FILE)

!ls /content/drive/MyDrive/dataset/

In [ ]:
!rm -rf dataset_root
!cp '/content/drive/MyDrive/{DATASET_ZIP}' ./
!unzip -q $DATASET_FILE
!mkdir dataset_root
!mv $DATASET_DIR './dataset_root'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms.v2 as transforms  # Updated to v2 to avoid deprecations
from torchvision.utils import save_image
from torch.utils.tensorboard import SummaryWriter  # Updated to native TensorBoard
from IPython.display import Image, display

# Added for perceptual loss (using pre-trained VGG)
from torchvision.models import vgg16, VGG16_Weights

# Added for LR scheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
bs = 64

# Added augmentations for better generalization (mild to avoid distorting road features)
transform = transforms.Compose([
    transforms.Resize((120, 160)),
    transforms.CenterCrop((80, 160)),  # Updated to CenterCrop for central region focus
    transforms.RandomHorizontalFlip(p=0.5),  # New: Helps with symmetry in driving data
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),  # Stronger: Simulates more lighting variations
    transforms.RandomRotation(10),  # New: Mild rotation for road variance
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # New: Center to [-1,1] for stability
])

dataset = datasets.ImageFolder(
    root='./dataset_root',
    transform=transform  # Updated to use the new transform
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=bs,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(len(dataset.imgs), len(dataloader))

In [ ]:
latent_dim = 32  # Reduced from 128 for better compression and to avoid sparse latent space

class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),  # 3x80x160 -> 32x40x80
            nn.ReLU(),
            nn.Dropout(0.1),  # New: Dropout for regularization
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # -> 64x20x40
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # -> 128x10x20
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # -> 256x5x10
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256 * 5 * 10, latent_dim)
        self.fc_logvar = nn.Linear(256 * 5 * 10, latent_dim)
        self.decoder_input = nn.Linear(latent_dim, 256 * 5 * 10)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 256x5x10 -> 128x10x20
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # -> 64x20x40
            nn.ReLU(),
            nn.Dropout(0.1),  # New: Dropout for regularization
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # -> 32x40x80
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),  # -> 3x80x160
            nn.Tanh(),  # Changed from Sigmoid to match Normalize [-1,1]
        )

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x = self.decoder_input(z)
        x = x.view(x.size(0), 256, 5, 10)
        x = self.decoder(x)
        return x

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights

# 1. Chuẩn bị VGG (Giữ nguyên phần này của bạn)
# Lưu ý: Với Jetson Nano, nếu bị OOM (tràn RAM), hãy cân nhắc đổi sang vgg11 hoặc chỉ lấy features[:4]
vgg = vgg16(weights=VGG16_Weights.DEFAULT).features[:16].eval().to(device)
for param in vgg.parameters():
    param.requires_grad = False

# 2. Định nghĩa Normalization chuẩn của ImageNet (BẮT BUỘC cho VGG)
# VGG không hiểu ảnh pixel 0-1, nó cần ảnh đã được chuẩn hóa theo mean/std này
vgg_normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                       std=[0.229, 0.224, 0.225])

def perceptual_loss(recon_x, x):
    # Đầu tiên, denormalize từ [-1,1] (do Tanh output và Normalize trong transforms) sang [0,1]
    x = (x + 1) / 2
    recon_x = (recon_x + 1) / 2

    # Normalize theo chuẩn ImageNet cho VGG
    x_norm = vgg_normalization(x)
    recon_x_norm = vgg_normalization(recon_x)

    feat_recon = vgg(recon_x_norm)
    feat_x = vgg(x_norm)

    # Dùng MSE cho feature space là chuẩn nhất
    return F.mse_loss(feat_recon, feat_x, reduction='mean')

def loss_function(recon_x, x, mu, logvar, beta=1.0, perc_weight=0.1):
    # --- THAY ĐỔI QUAN TRỌNG 1: Đổi BCE sang MSE ---
    # MSE phù hợp hơn cho ảnh màu continuous từ camera.
    # reduction='sum' để giữ độ lớn tương đương logic cũ của bạn
    REC = F.mse_loss(recon_x, x, reduction='sum')

    # --- KLD (Giữ nguyên) ---
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # --- THAY ĐỔI QUAN TRỌNG 2: Điều chỉnh Perceptual ---
    # Không nhân với numel() nữa vì MSE 'sum' đã rất lớn rồi.
    # Perceptual loss nên hoạt động như một "hướng dẫn" phụ trợ.
    p_loss = perceptual_loss(recon_x, x)

    # Scale PERC lên một chút để nó không quá bé so với REC (MSE sum)
    # Vì feature MSE thường rất nhỏ (vd: 0.005), cần nhân hệ số để tác động được vào Gradient
    # Với ảnh 80x160, MSE sum thường tầm 1000-3000. Cần PERC tầm 100-500.
    # Bạn có thể tinh chỉnh hệ số 10000 này dựa trên log training (ví dụ, nếu PERC quá nhỏ, tăng lên; nếu lấn át REC, giảm xuống)
    PERC = p_loss * 10000  # Hệ số scaling thủ công, có thể tinh chỉnh

    # Tổng hợp loss
    return REC + beta * KLD + perc_weight * PERC, REC

In [ ]:
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
writer = SummaryWriter()

In [ ]:
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [ ]:
model.train()
fixed_batch = next(iter(dataloader))[0].to(device)  # Fixed batch for embedding visualization
for epoch in range(100):
    train_loss = 0.0
    rec_loss = 0.0
    beta = min(0.1 + epoch * 0.05, 2.0)  # Gentler beta ramp, max 2.0
    perc_weight = 0.05  # Keep, but lower if PERC dominates logs
    for batch_idx, (data, _) in enumerate(dataloader):
        data = data.to(device)
        recon_batch, mu, logvar = model(data)
        loss, REC = loss_function(recon_batch, data, mu, logvar, beta, perc_weight)
        optimizer.zero_grad()
        loss.backward()
        train_loss += loss.item()
        rec_loss += REC.item()
        optimizer.step()
    scheduler.step(loss.item())  # Step scheduler based on last batch loss (or avg if preferred)
    print('====> Epoch: {} Average loss: {:.4f}'.format(epoch, train_loss / len(dataloader.dataset)))
    writer.add_scalar('loss/train', train_loss / len(dataloader.dataset), epoch)
    writer.add_scalar('loss/rec', rec_loss / len(dataloader.dataset), epoch)
    # Enable projector: Log latent embeddings every 10 epochs
    if epoch % 10 == 0:
        with torch.no_grad():
            _, mu_fixed, _ = model(fixed_batch)
            writer.add_embedding(mu_fixed, metadata=fixed_batch.data.cpu(), label_img=fixed_batch.cpu(), global_step=epoch, tag='latent_space')

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/vae_improved_0002.torch')

In [ ]:
%tensorboard --logdir runs